In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline

# 1. Load and clean your data
data = pd.read_csv('/content/feeds.csv')
data = data.dropna(axis=1, how='all')
data = data.drop_duplicates()

# 2. Fill missing feature values with random realistic demo values
def fill_with_random(df, col, vmin, vmax):
    if col in df.columns:
        mask = df[col].isnull()
        df.loc[mask, col] = np.random.uniform(vmin, vmax, size=mask.sum())


fill_with_random(data, 'field1', 30, 42)
fill_with_random(data, 'field2', 30, 90)
fill_with_random(data, 'field3', 60, 120)     # Heartbeat (bpm)
fill_with_random(data, 'latitude', 10.0, 30.0)   # Location latitude
fill_with_random(data, 'longitude', 70.0, 90.0)  # Location longitude


# Health risk: high temp OR high heartbeat
data['health_risk'] = (
    (data['field1'] > data['field1'].median()) |
    (data['field3'] > data['field3'].median())
).astype(int)

# Hygiene risk: high humidity
data['hygiene_risk'] = (data['field2'] > data['field2'].median()).astype(int)

# Environmental risk: high temperature
data['environment_risk'] = (data['field1'] > data['field1'].median()).astype(int)

# 4. Prepare features and targets
requested_features = ['field1', 'field2', 'field3', 'latitude', 'longitude']
feature_cols = [col for col in requested_features if col in data.columns]
X = data[feature_cols]

y_health = data['health_risk']
y_hygiene = data['hygiene_risk']
y_env = data['environment_risk']

# 5. Train-test split (same indices for all targets)
X_train, X_test, idx_train, idx_test = train_test_split(X, range(len(X)), test_size=0.25, random_state=42)
y_health_train = y_health.iloc[idx_train]
y_health_test = y_health.iloc[idx_test]
y_hygiene_train = y_hygiene.iloc[idx_train]
y_hygiene_test = y_hygiene.iloc[idx_test]
y_env_train = y_env.iloc[idx_train]
y_env_test = y_env.iloc[idx_test]

# 6. Define pipeline factory for each risk
def make_pipe():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(C=1, kernel='rbf', gamma='scale'))
    ])

# 7. Train the models
health_pipe = make_pipe()
hygiene_pipe = make_pipe()
env_pipe = make_pipe()

health_pipe.fit(X_train, y_health_train)
hygiene_pipe.fit(X_train, y_hygiene_train)
env_pipe.fit(X_train, y_env_train)

# 8. Predict and report for all risk types
print("==== HEALTH RISK ====")
health_pred = health_pipe.predict(X_test)
print("Accuracy:", accuracy_score(y_health_test, health_pred))
print("Confusion Matrix:\n", confusion_matrix(y_health_test, health_pred))
print("Report:\n", classification_report(y_health_test, health_pred))

print("==== HYGIENE RISK ====")
hygiene_pred = hygiene_pipe.predict(X_test)
print("Accuracy:", accuracy_score(y_hygiene_test, hygiene_pred))
print("Confusion Matrix:\n", confusion_matrix(y_hygiene_test, hygiene_pred))
print("Report:\n", classification_report(y_hygiene_test, hygiene_pred))

print("==== ENVIRONMENTAL RISK ====")
env_pred = env_pipe.predict(X_test)
print("Accuracy:", accuracy_score(y_env_test, env_pred))
print("Confusion Matrix:\n", confusion_matrix(y_env_test, env_pred))
print("Report:\n", classification_report(y_env_test, env_pred))



==== HEALTH RISK ====
Accuracy: 0.967741935483871
Confusion Matrix:
 [[ 6  0]
 [ 1 24]]
Report:
               precision    recall  f1-score   support

           0       0.86      1.00      0.92         6
           1       1.00      0.96      0.98        25

    accuracy                           0.97        31
   macro avg       0.93      0.98      0.95        31
weighted avg       0.97      0.97      0.97        31

==== HYGIENE RISK ====
Accuracy: 0.967741935483871
Confusion Matrix:
 [[19  1]
 [ 0 11]]
Report:
               precision    recall  f1-score   support

           0       1.00      0.95      0.97        20
           1       0.92      1.00      0.96        11

    accuracy                           0.97        31
   macro avg       0.96      0.97      0.97        31
weighted avg       0.97      0.97      0.97        31

==== ENVIRONMENTAL RISK ====
Accuracy: 0.967741935483871
Confusion Matrix:
 [[13  0]
 [ 1 17]]
Report:
               precision    recall  f1-score   s